In [12]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS


iedb_II_conf = (
    pl.read_parquet("../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_II/triad/iedb_II_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_II_tcrdock = pl.read_parquet(
    "../../data/iedb_II/triad/staged/iedb_II_triad.af3_tcrdock.parquet"
)

iedb_II = iedb_II_conf.join(
    iedb_II_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

template_dgeom = pl.read_csv(
    "../../data/pdb/raw/ternary_templates_v2.tsv", separator="\t"
).with_columns(
    pl.struct(
        **{
            k: pl.col(k)
            for k in [
                "d",
                "torsion",
                "mhc_unit_x_is_negative",
                "tcr_unit_y",
                "tcr_unit_z",
                "tcr_unit_x_is_negative",
                "mhc_unit_y",
                "mhc_unit_z",
            ]
        }
    ).alias("dgeom"),
    pl.when(pl.col("mhc_class") == 1)
    .then(pl.lit("I"))
    .otherwise(pl.lit("II"))
    .alias("mhc_class"),
)

class_II_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "II").select("dgeom").to_series()
)

class_II_distr = mn_distr_from_dgeom_ndarr(class_II_t_dgeom)

_, iedb_II_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_II.select("pred_dgeom_4").to_series()),
    *class_II_distr,
)

iedb_II = iedb_II.with_columns(pl.Series(name="p_dgeom", values=iedb_II_p_dgeom))

iedb_II = iedb_II.explode("receptor_id", "references")


iedb_I_conf = (
    pl.read_parquet("../../data/iedb_I/triad/iedb_I_triad.conf_af3.parquet")
    .drop(["receptor_id", "references"])
    .join(
        pl.read_parquet(
            "../../data/iedb_I/triad/iedb_I_triad.receptor_reference.parquet"
        ).select("job_name", "receptor_id", "references"),
        on="job_name",
        how="left",
    )
)
iedb_I = iedb_I_conf
iedb_I_tcrdock = pl.read_parquet(
    "../../data/iedb_I/triad/staged/iedb_I_triad.af3_tcrdock.parquet"
)

iedb_I = iedb_I_conf.join(
    iedb_I_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

class_I_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "I").select("dgeom").to_series()
)

class_I_distr = mn_distr_from_dgeom_ndarr(class_I_t_dgeom)

_, iedb_I_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_I.select("pred_dgeom_4").to_series()),
    *class_I_distr,
)

iedb_I = iedb_I.with_columns(pl.Series(name="p_dgeom", values=iedb_I_p_dgeom))

iedb_I = iedb_I.explode("receptor_id", "references")

# REMOVEME TO ADD BACK IN
iedb_II_annot = pl.read_parquet(
    "../../data/iedb_II_full/triad/iedb_II_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))

iedb_II = iedb_II.join(iedb_II_annot, on="job_name", how="anti")

# REMOVEME TO ADD BACK IN
iedb_I_annot = pl.read_parquet(
    "../../data/iedb_I_full/triad/iedb_I_triad.annotated.parquet"
).filter(pl.col("triad_in_pdb"))

iedb_I = iedb_I.join(iedb_I_annot, on="job_name", how="anti")

# additionally filter out
iedb_I = iedb_I.filter(pl.col("job_name") != "121f50b609621ebe92777820dad90eb9")

docking_feats = [
    "d",
    "mhc_unit_y",
    "mhc_unit_z",
    "tcr_unit_y",
    "tcr_unit_z",
    "torsion",
    "p_dgeom",
]

interface_feats = [
    "mean_p_tcr_pae",
    "mean_tcr_p_pae",
    "mean_mhc_tcr_pae",
    "mean_tcr_mhc_pae",
    "mean_p_tcr_contact_prob",
    "mean_tcr_p_contact_prob",
    "mean_mhc_tcr_contact_prob",
    "mean_tcr_mhc_contact_prob",
    "mean_p_tcr_interface_pae",
    "mean_tcr_p_interface_pae",
    "mean_tcr_pmhc_interface_pae",
    "mean_pmhc_tcr_interface_pae",
    "mean_p_tcr_interface_contact_prob",
    "mean_tcr_p_interface_contact_prob",
    "mean_tcr_pmhc_interface_contact_prob",
    "mean_pmhc_tcr_interface_contact_prob",
    "mean_p_mhc_pae",
    "mean_mhc_p_pae",
    "mean_mhc_p_interface_pae",
    "mean_p_mhc_interface_pae",
    # "mean_mhc_p_contact_prob",
    # "mean_p_mhc_contact_prob",
    "min_p_tcr_pae",
    "min_mhc_tcr_pae",
    "min_tcr_p_pae",
    "min_tcr_mhc_pae",
    "tcr_mhc_contacts",
    "tcr_p_contacts",
]


local_feats_II = [
    "peptide_mean_pLDDT",
    "peptide_mean_pLDDT_II",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]


local_feats_I = [
    "peptide_mean_pLDDT",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]

summary_feats = [
    "iptm",
    "ptm",
    "ranking_score",
]

featnames_II = docking_feats + interface_feats + local_feats_II + summary_feats
feat_type_II = (
    ["docking"] * len(docking_feats)
    + ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats_II)
    + ["summary"] * len(summary_feats)
)
featnames_I = docking_feats + interface_feats + local_feats_I + summary_feats
feat_type_I = (
    ["docking"] * len(docking_feats)
    + ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats_I)
    + ["summary"] * len(summary_feats)
)

In [13]:
all_dat = pl.concat([iedb_I, iedb_II], how="align")

In [23]:
all_dat.explode("references").explode("receptor_id")

job_name,cognate,peptide,mhc_class,mhc_1_chain,mhc_1_species,mhc_1_name,mhc_1_seq,mhc_2_chain,mhc_2_species,mhc_2_name,mhc_2_seq,tcr_1_chain,tcr_1_species,tcr_1_seq,tcr_2_chain,tcr_2_species,tcr_2_seq,tcr_1_cdr_1,tcr_1_cdr_2,tcr_1_cdr_2_5,tcr_1_cdr_3,tcr_2_cdr_1,tcr_2_cdr_2,tcr_2_cdr_2_5,tcr_2_cdr_3,pmhc_in_validation,chain_iptm,chain_pair_iptm,chain_pair_pae_min,chain_ptm,fraction_disordered,has_clash,iptm,ptm,ranking_score,mean_p_tcr_interface_pae,…,tcr_mhc_contacts,tcr_p_contacts,contact_map,peptide_mean_pLDDT,tcr_1_cdr_1_mean_pLDDT,tcr_1_cdr_2_mean_pLDDT,tcr_1_cdr_2_5_mean_pLDDT,tcr_1_cdr_3_mean_pLDDT,tcr_2_cdr_1_mean_pLDDT,tcr_2_cdr_2_mean_pLDDT,tcr_2_cdr_2_5_mean_pLDDT,tcr_2_cdr_3_mean_pLDDT,tcr_cdrs_mean_pLDDT,mhc_helices_mean_pLDDT,mean_p_mhc_pae,mean_mhc_p_pae,mean_mhc_p_interface_pae,mean_p_mhc_interface_pae,mean_mhc_p_interface_contact_prob,mean_p_mhc_interface_contact_prob,min_p_tcr_pae,min_mhc_tcr_pae,min_tcr_p_pae,min_tcr_mhc_pae,receptor_id,references,pred_dgeom_4,d,mhc_unit_x_is_negative,mhc_unit_y,mhc_unit_z,tcr_unit_x_is_negative,tcr_unit_y,tcr_unit_z,torsion,p_dgeom,peptide_mean_pLDDT_II
str,bool,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,list[f64],list[list[f64]],list[list[f64]],list[f64],f64,f64,f64,f64,f64,f64,…,i64,i64,list[list[f64]],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,struct[8],f64,bool,f64,f64,bool,f64,f64,f64,f64,f64
"""0001000d9abfb83446cf9ae5a6d899…",false,"""RQWGPDPAAV""","""I""","""heavy""","""human""","""A*02:01""","""GSHSMRYFFTSVSRPGRGEPRFIAVGYVDD…","""light""","""human""","""B2M""","""MSRSVALAVLALLSLSGLEAIQRTPKIQVY…","""alpha""","""human""","""QKEVEQNSGPLSVPEGAIASLNCTYSDRGS…","""beta""","""human""","""NAGVTQTPKFQVLKTGQSMTLQCAQDMNHE…","""DRGSQS""","""IYSNGD""","""NKASQY""","""CAVTTDSWGKLQF""","""MNHEY""","""SVGAGI""","""STTED""","""CASRPGLAGGRPEQYF""",null,"[0.72, 0.83, … 0.68]","[[0.02, 0.88, … 0.59], [0.88, 0.88, … 0.71], … [0.59, 0.71, … 0.86]]","[[0.76, 1.35, … 3.89], [1.14, 0.76, … 3.09], … [3.65, 2.92, … 0.76]]","[0.02, 0.88, … 0.86]",0.07,0.0,0.85,0.86,0.89,6.3056,…,3,21,"[[0.0, 0.0, … 0.0], [1.0, 0.0, … 1.0], … [0.0, 0.0, … 0.0]]",83.81,87.815454,90.595,91.976326,87.622277,87.530213,88.287941,91.631081,81.995714,87.319686,91.311422,9.64045,5.382148,2.563735,6.191832,0.057108,0.057108,2.69,2.14,2.44,2.36,null,null,"{30.698021,false,0.061153,-0.135127,false,0.078599,0.042181,3.453654}",30.698021,false,0.061153,-0.135127,false,0.078599,0.042181,3.453654,0.879428,null
"""00013202cb1b174307649eac4179b5…",false,"""FVNLEQHVV""","""I""","""heavy""","""human""","""A*02:01""","""GSHSMRYFFTSVSRPGRGEPRFIAVGYVDD…","""light""","""human""","""B2M""","""MSRSVALAVLALLSLSGLEAIQRTPKIQVY…","""alpha""","""human""","""ILNVEQSPQSLHVQEGDSTNFTCSFPSSNF…","""beta""","""human""","""GAGVSQSPSNKVTEKGKDVELRCDPISGHT…","""SSNFYA""","""MTLNGDE""","""NTKEGY""","""CARNAGNMLTF""","""SGHTA""","""FQGNSA""","""TGGSV""","""CASSPVLAPETQYF""",null,"[0.51, 0.56, … 0.34]","[[0.02, 0.93, … 0.16], [0.93, 0.88, … 0.21], … [0.16, 0.21, … 0.86]]","[[0.76, 1.3, … 16.110001], [1.01, 0.76, … 14.28], … [14.09, 11.26, … 0.76]]","[0.02, 0.88, … 0.86]",0.07,0.0,0.63,0.67,0.67,19.693834,…,0,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]",77.564415,68.866875,76.864231,85.561428,79.062099,78.364375,73.006744,78.578929,80.063429,77.97468,86.494471,6.853419,4.798493,1.923,3.711764,0.059458,0.059458,16.110001,14.28,14.09,11.26,null,null,"{30.25337,false,-0.051024,0.101736,false,-0.301442,-0.107776,1.757227}",30.25337,false,-0.051024,0.101736,false,-0.301442,-0.107776,1.757227,9.6512e-12,null
"""0001377648c00f57063ba72ea74d4c…",false,"""MYTEMLKSI""","""I""","""heavy""","""human""","""A*24:02""","""GSHSMRYFSTSVSRPGRGEPRFIAVGYVDD…","""light""","""human""","""B2M""","""MSRSVALAVLALLSLSGLEAIQRTPKIQVY…","""alpha""","""human""","""DAKTTQPNSMESNEEEPVHLPCNHSTISGT…","""beta""","""human""","""DARVTQTPRHKVTEMGQEVTMRCQPILGHN…","""TIS

In [25]:
from tcrtrifold.utils import FORMAT_COLS, TCRDIST_COLS

all_dat_fmt = (
    all_dat.explode("references")
    .explode("receptor_id")
    .select(FORMAT_COLS + TCRDIST_COLS + featnames_II)
    .sort(by=["mhc_class", "cognate", "peptide"], descending=True)
)

all_dat_fmt.write_csv("supp_table_1.tsv", separator="\t")